# iNDI metadata processing
The iNDI metadata is pretty broken up, I'm trying to unify into a cleaner metadata sheet for downstream analysis, this should be cleaned up a lot better down the road and processed my thuroughly from the original metadata sheets though

In [30]:
import pathlib as Path
import pandas as pd

## Set up paths

In [31]:
META_DIR = "/data/CARDPB2/iNDI/Production/metadata"

AUTO_FN = f"{META_DIR}/all_automated_plates_combined.csv"
MAN_FN  = f"{META_DIR}/all_manual_plates_combined.csv"

JAX_FN  = f"{META_DIR}/JAX_iPSC_Catalog.tsv"
PLATE_ID_FN = f"{META_DIR}/indi_plateID_to_folderID.csv"
ASP_POS_ID_FN = f"{META_DIR}/asp_pos_ids_metadata.csv"
BATCH_FN = f"{META_DIR}/AbPanel1/batches.csv"

print(JAX_FN, PLATE_ID_FN, ASP_POS_ID_FN, BATCH_FN, sep="\n")

/data/CARDPB2/iNDI/Production/metadata/JAX_iPSC_Catalog.tsv
/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv
/data/CARDPB2/iNDI/Production/metadata/asp_pos_ids_metadata.csv
/data/CARDPB2/iNDI/Production/metadata/AbPanel1/batches.csv


## Get JAX IDs for manual data

In [32]:
MAN = pd.read_csv(MAN_FN)
JAX = pd.read_csv(JAX_FN, sep="\t")

MAN[["Gene_anno", "Variant_anno", "Genotype_anno"]] = MAN["Genotype_Name"].str.split(" ", n=2, expand=True)

MAN["Gene_anno"] = MAN["Gene_anno"].replace({"Parental": "KOLF2.1J"})
MAN["Gene_anno"] = MAN["Gene_anno"].replace({"SCNB": "SNCB"})
MAN["Variant_anno"] = MAN["Variant_anno"].replace({"R317W": "R406W"})

# TBK1 variant fix
MAN.loc[
    (MAN["Gene_anno"] == "TBK1") & (MAN["Variant_anno"] == "E696"),
    "Variant_anno",
] = "E696K"

# UBQLN2 P506T genotype overrides (based on original HET/REV codes)
ubqln2_p506t = (MAN["Gene_anno"] == "UBQLN2") & (MAN["Variant_anno"] == "P506T")
MAN.loc[ubqln2_p506t & (MAN["Genotype_anno"] == "HET"), "Genotype_anno"] = "SNV/Y"
MAN.loc[ubqln2_p506t & (MAN["Genotype_anno"] == "REV"), "Genotype_anno"] = "REV/Y"

# General genotype mapping for everything else
MAN["Genotype_anno"] = MAN["Genotype_anno"].replace({
    "HET": "SNV/WT",
    "HOM": "SNV/SNV",
    "REV": "REV/WT",
})

MAN = MAN.merge(
    JAX,
    left_on=["Gene_anno", "Variant_anno", "Genotype_anno"],
    right_on=["Gene", "Variant", "Genotype"],
    how="left",
)

# Drop *_anno columns
MAN = MAN.drop(columns=["Gene_anno", "Variant_anno", "Genotype_anno"])
MAN = MAN.rename(columns={"JAX_Code": "JAX_ID"})

print(MAN.head())
print(MAN.columns.to_list())

  iNDI_Plate_ID Well_ID Genotype_Name Batch_Info       JAX_ID      Gene  \
0     INDIEL001     A01      Parental        NaN  JIPSC001000  KOLF2.1J   
1     INDIEL001     A02      Parental        NaN  JIPSC001000  KOLF2.1J   
2     INDIEL001     A03      Parental        NaN  JIPSC001000  KOLF2.1J   
3     INDIEL001     A04      Parental        NaN  JIPSC001000  KOLF2.1J   
4     INDIEL001     A05      Parental        NaN  JIPSC001000  KOLF2.1J   

  Variant Genotype Gene_Var_Type  
0     NaN      NaN      KOLF2.1J  
1     NaN      NaN      KOLF2.1J  
2     NaN      NaN      KOLF2.1J  
3     NaN      NaN      KOLF2.1J  
4     NaN      NaN      KOLF2.1J  
['iNDI_Plate_ID', 'Well_ID', 'Genotype_Name', 'Batch_Info', 'JAX_ID', 'Gene', 'Variant', 'Genotype', 'Gene_Var_Type']


## Process automated metadata

In [33]:
AUTO = pd.read_csv(AUTO_FN)

AUTO["JAX_ID"] = AUTO["JAX_ID"].replace({"JIPSC1000": "JIPSC001000"})
AUTO = AUTO.drop(columns=["JAX_Code"])

print(AUTO.columns.to_list())
print(MAN.columns.to_list())

AUTO["Source"] = "automated"
MAN["Source"] = "manual"
COMBINED = pd.concat([AUTO, MAN], ignore_index=True)
print(COMBINED.head)
print(COMBINED.columns.to_list())

['iNDI_Plate_ID', 'Well_ID', 'Genotype_Name', 'JAX_ID', 'Batch_Info', 'Gene', 'Variant', 'Genotype', 'Gene_Var_Type', 'Cell_Line_Info']
['iNDI_Plate_ID', 'Well_ID', 'Genotype_Name', 'Batch_Info', 'JAX_ID', 'Gene', 'Variant', 'Genotype', 'Gene_Var_Type']
<bound method NDFrame.head of      iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info  \
0       INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN   
1       INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN   
2       INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN   
3       INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN   
4       INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN   
...            ...     ...                ...          ...        ...   
8421     INDIRK008     P20                NaN          NaN        NaN   
8422     INDIRK008     P21                NaN          NaN        NaN   
8423     INDIRK008     P22                NaN          NaN 

## Process Asp PosID sheet

In [34]:
ASP_POS_ID = pd.read_csv(ASP_POS_ID_FN)
ASP_POS_ID = ASP_POS_ID.drop(columns=["Disp PosID", "Disp_Pos_ID"])

print(ASP_POS_ID.head())

COMBINED_ASP = COMBINED.merge(
    ASP_POS_ID,
    left_on=["iNDI_Plate_ID", "Well_ID"],
    right_on=["iNDI_Plate_ID", "Well_ID"],
    how="left",
)

print(COMBINED_ASP.head())
print(COMBINED_ASP.columns.to_list())

  Asp PosID iNDI_Plate_ID Well_ID
0        A5    INDI00041D     A03
1        A5    INDI00041D     D03
2        A5    INDI00042D     A03
3        A5    INDI00042D     D03
4        A5    INDI00043D     A03
  iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info     Gene  \
0    INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
1    INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
2    INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN  HNRNPA1   
3    INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN  HNRNPA1   
4    INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN     PFN1   

  Variant Genotype          Gene_Var_Type  \
0   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
1   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
2   D262N   REV/WT   HNRNPA1_D262N_REV/WT   
3   D262N  SNV/SNV  HNRNPA1_D262N_SNV/SNV   
4    C71G   SNV/WT       PFN1_C71G_SNV/WT   

                                      Cell_Line_In

## Process batch sheet

In [35]:
BATCH = pd.read_csv(BATCH_FN)
print(BATCH.columns.to_list())

COMBINED_ASP["AbPanel"] = 1

COMBINED_ASP_BATCH = COMBINED_ASP.merge(
    BATCH,
    left_on=["iNDI_Plate_ID", "AbPanel"],
    right_on=["iNDI_Plate_ID", "AbPanel"],
    how="left",
)
print(COMBINED_ASP_BATCH.head)

['iNDI_Plate_ID', 'AbPanel', 'Batch']
<bound method NDFrame.head of      iNDI_Plate_ID Well_ID      Genotype_Name       JAX_ID Batch_Info  \
0       INDI00026D     A01  HNRNPA1 D262N HOM  JIPSC001038        NaN   
1       INDI00026D     A02  HNRNPA1 D262N HOM  JIPSC001038        NaN   
2       INDI00026D     A03  HNRNPA1 D262N REV  JIPSC001042        NaN   
3       INDI00026D     A04  HNRNPA1 D262N HOM  JIPSC001038        NaN   
4       INDI00026D     A05      PFN1 C71G HET  JIPSC001034        NaN   
...            ...     ...                ...          ...        ...   
8421     INDIRK008     P20                NaN          NaN        NaN   
8422     INDIRK008     P21                NaN          NaN        NaN   
8423     INDIRK008     P22                NaN          NaN        NaN   
8424     INDIRK008     P23                NaN          NaN        NaN   
8425     INDIRK008     P24                NaN          NaN        NaN   

         Gene Variant Genotype          Gene_Var_Type  

## Process plate ID sheet

In [36]:
PLATE_ID = pd.read_csv(PLATE_ID_FN)

print(PLATE_ID.head)
print(PLATE_ID.columns.to_list())

COMBINED_ASP_BATCH_ID = COMBINED_ASP_BATCH.merge(
    PLATE_ID,
    left_on=["iNDI_Plate_ID"],
    right_on=["iNDI_Plate_ID"],
    how="left",
)
print(COMBINED_ASP_BATCH_ID.head)

<bound method NDFrame.head of    iNDI_Plate_ID         Layout                             Folder_ID
0     INDI00002D       Plate002  2b1c31ad-579d-4ea6-a782-251ea083ed6a
1     INDI00007D       Plate007  d7e04b5c-a253-42ff-859f-2af0e6047a00
2     INDI00011D       Plate011  f8062781-891b-4688-a59e-fccbf48325cf
3     INDI00015D       Plate015  caef83c7-6b44-4b64-b8ec-e8d84ca6759d
4     INDI00019D       Plate019  c8f2dbbe-6632-4063-ab68-0b4fa37909af
5     INDI00023D       Plate023  32d75d22-bd17-4629-8257-a4056e3ffcee
6     INDI00026D       Plate026  c6a92be6-a13a-4957-a62d-9f99961c903f
7     INDI00035D       Plate002  c170959e-633a-458c-8542-2aa619c0d7f8
8     INDI00039D       Plate039  07fc5da6-9d7d-4c97-858b-4b76df1859a5
9     INDI00043D       Plate043  4405a3b2-6b88-49b1-91f3-992e09ccbd16
10    INDI00047D       Plate047  72c0a713-b5d5-4cd4-a826-c556368ed1d1
11    INDI00049D       Plate049  7e03b502-ad64-4090-a9e2-fc4c3ecdae11
12    INDI00053D       Plate053  dad5d97b-1a66-4a2e-9ea2-bde

## Write final metadata sheet

In [37]:
COMBINED_ASP_BATCH_ID.to_csv("COMBINED_ASP_BATCH_METADATA.csv", index=False)